# Module 02, Notebook 3: Posterior Predictive Checks with ArviZ

## Learning Objectives
By completing this notebook, you will:
1. Generate and interpret posterior predictive distributions for ITS models
2. Use multiple PPC test quantities to detect model misspecification
3. Visualize the counterfactual distribution with ArviZ
4. Identify when a model needs to be refined (seasonal pattern, non-linear trend, heavy tails)
5. Interpret Bayesian p-values (posterior predictive p-values)

## Posterior Predictive Checks (PPC)

A posterior predictive check asks: "If my model is correct, does the data it would generate look like the data I actually observed?"

The workflow:
1. Sample parameters from the posterior
2. Generate new data from the likelihood using those parameters
3. Compare generated data statistics to observed data statistics
4. If they differ systematically, the model is misspecified

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["Generate and interpret posterior predictive distributions for ITS models", "Use multiple PPC test quantities to detect model misspecification", "Visualize the counterfactual distribution with ArviZ", "Identify when a model needs to be refined (seasonal pattern, non-linear trend, heavy tails)", "Interpret Bayesian p-values (posterior predictive p-values)", "Sample parameters from the posterior"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import causalpy as cp
from scipy import stats

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

In [ ]:
# Dataset and model — smoking ban ITS with seasonal controls

np.random.seed(42)
n_months = 48
n_pre = 24

months = np.arange(n_months).astype(float)
calendar_month = months % 12
seasonal_effect = 6.0 * np.cos(2 * np.pi * calendar_month / 12)
treated = (months >= n_pre).astype(float)
t_post = np.maximum(months - n_pre, 0).astype(float)

ami_rate = (
    85.0 - 0.15 * months - 12.0 * treated - 0.10 * t_post * treated
    + seasonal_effect
    + np.random.normal(0, 4.0, n_months)
)

df = pd.DataFrame({
    "month": months,
    "ami_rate": ami_rate,
    "treated": treated,
    "t_post": t_post,
    "calendar_month": calendar_month.astype(int),
})

# Fit seasonally-adjusted model
model_seasonal = cp.pymc_experiments.InterruptedTimeSeries(
    data=df,
    treatment_time=n_pre,
    formula="ami_rate ~ 1 + month + treated + t_post + C(calendar_month)",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 1000, "tune": 1000, "chains": 4,
                       "progressbar": True, "random_seed": 42}
    ),
)

print("Seasonally adjusted model fitted.")

## 1. Visual PPC: Observed vs Posterior Predictive

In [ ]:
# Sample posterior predictive

with model_seasonal.model:
    ppc_samples = pm.sample_posterior_predictive(
        model_seasonal.idata,
        random_seed=42,
    )

# Extract predictions: shape (chains, draws, n_obs)
y_ppc = ppc_samples.posterior_predictive["y_hat"].values
y_ppc_flat = y_ppc.reshape(-1, n_months)  # (n_samples, n_obs)

print(f"PPC samples shape: {y_ppc_flat.shape}")

In [ ]:
# Visual PPC: time-series overlay
# Blue band = posterior predictive distribution
# Red dots = observed data

ppc_mean = y_ppc_flat.mean(axis=0)
ppc_lower = np.percentile(y_ppc_flat, 3, axis=0)   # 3rd percentile
ppc_upper = np.percentile(y_ppc_flat, 97, axis=0)  # 97th percentile
ppc_q10 = np.percentile(y_ppc_flat, 10, axis=0)
ppc_q90 = np.percentile(y_ppc_flat, 90, axis=0)

fig, ax = plt.subplots(figsize=(13, 5))

# 94% PPC band
ax.fill_between(months, ppc_lower, ppc_upper, alpha=0.2, color="#3498db", label="94% PPC")
# 80% PPC band
ax.fill_between(months, ppc_q10, ppc_q90, alpha=0.3, color="#3498db", label="80% PPC")
# PPC mean
ax.plot(months, ppc_mean, color="#2980b9", linewidth=2, label="PPC mean")

# Observed data
ax.scatter(months, ami_rate, color="#e74c3c", s=25, zorder=5, label="Observed")

ax.axvline(n_pre, color="#27ae60", linestyle="--", linewidth=2, label="Intervention")
ax.set_title("Posterior Predictive Check: Seasonally Adjusted ITS Model", fontsize=13)
ax.set_xlabel("Month")
ax.set_ylabel("AMI Rate per 100,000")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Fraction of observations within the 94% band
within_94 = ((ami_rate >= ppc_lower) & (ami_rate <= ppc_upper)).mean()
print(f"Fraction of observations within 94% PPC band: {within_94:.1%}")
print(f"Expected: ~94%. If much lower, the model underestimates variance.")

## 2. Test Quantity PPCs

Check specific statistics: if the model is correct, the observed statistic should look like a typical draw from the posterior predictive distribution of that statistic.

In [ ]:
# Bayesian p-values for key test quantities
# Bayesian p-value = P(T(y_rep) > T(y_obs) | y_obs)
# Values near 0.5 = model correctly captures this statistic
# Values near 0 or 1 = model systematically over/under-predicts this statistic

def compute_bayesian_pvalue(y_obs, y_ppc_flat, test_fn, name):
    """Compute Bayesian p-value for a test statistic."""
    t_obs = test_fn(y_obs)
    t_rep = np.array([test_fn(y_ppc_flat[i]) for i in range(y_ppc_flat.shape[0])])
    p_val = (t_rep >= t_obs).mean()
    return t_obs, t_rep, p_val, name


# Define test statistics
test_statistics = [
    (lambda y: y.mean(), "Mean"),
    (lambda y: y.std(), "Std Dev"),
    (lambda y: y.max(), "Maximum"),
    (lambda y: y.min(), "Minimum"),
    (lambda y: np.percentile(y, 95), "95th Percentile"),
    (lambda y: np.percentile(y, 5), "5th Percentile"),
    # Autocorrelation at lag 1
    (lambda y: np.corrcoef(y[:-1], y[1:])[0, 1], "Lag-1 Autocorrelation"),
]

results = []
for test_fn, name in test_statistics:
    t_obs, t_rep, p_val, _ = compute_bayesian_pvalue(ami_rate, y_ppc_flat, test_fn, name)
    results.append({
        "Statistic": name,
        "Observed": t_obs,
        "PPC Mean": t_rep.mean(),
        "Bayesian p-value": p_val,
        "Assessment": "OK" if 0.05 <= p_val <= 0.95 else "FLAG"
    })

results_df = pd.DataFrame(results)
print("Posterior Predictive Check: Test Statistics")
print("=" * 70)
print(results_df.to_string(index=False, float_format="{:.3f}".format))
print()
print("Bayesian p-value near 0.5 = model captures this statistic well")
print("Bayesian p-value < 0.05 = model systematically underestimates this statistic")
print("Bayesian p-value > 0.95 = model systematically overestimates this statistic")

In [ ]:
# Visual PPC for test statistics: kernel density overlay

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, (test_fn, name) in zip(axes[:6], test_statistics[:6]):
    t_obs, t_rep, p_val, _ = compute_bayesian_pvalue(ami_rate, y_ppc_flat, test_fn, name)

    # Plot posterior predictive distribution of the statistic
    ax.hist(t_rep, bins=40, edgecolor="white", color="#3498db", alpha=0.8, density=True)
    ax.axvline(t_obs, color="#e74c3c", linewidth=2.5, label=f"Observed: {t_obs:.1f}")
    ax.set_title(f"{name}\nBayesian p-value = {p_val:.3f}", fontsize=11)
    ax.set_xlabel(name)
    ax.legend(fontsize=9)

plt.suptitle("PPC Test Quantities: Observed (red) vs Posterior Predictive Distribution (blue)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. PPC for Model Comparison: Seasonal vs Naive

In [ ]:
# Compare PPC for seasonally adjusted vs naive model
# The seasonal model should fit better

# Fit the naive (no seasonal) model
model_naive = cp.pymc_experiments.InterruptedTimeSeries(
    data=df,
    treatment_time=n_pre,
    formula="ami_rate ~ 1 + month + treated + t_post",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 500, "tune": 500, "chains": 4,
                       "progressbar": False, "random_seed": 42}
    ),
)

# Sample PPC for naive model
with model_naive.model:
    ppc_naive = pm.sample_posterior_predictive(model_naive.idata, random_seed=42)

y_ppc_naive = ppc_naive.posterior_predictive["y_hat"].values.reshape(-1, n_months)

# Compare residual variance (how well do the models capture variability)
sigma_seasonal = model_seasonal.idata.posterior["sigma"].values.flatten().mean()
sigma_naive = model_naive.idata.posterior["sigma"].values.flatten().mean()

print("Residual Noise Comparison (lower = better fit)")
print(f"  Naive model (σ):              {sigma_naive:.2f}")
print(f"  Seasonally adjusted (σ):      {sigma_seasonal:.2f}")
print(f"  Reduction in residual noise:  {(sigma_naive - sigma_seasonal) / sigma_naive:.1%}")

In [ ]:
# LOO-CV model comparison

comparison = az.compare(
    {"seasonal": model_seasonal.idata, "naive": model_naive.idata},
    ic="loo",
)

print("LOO Cross-Validation Comparison")
print(comparison[["elpd_loo", "p_loo", "elpd_diff", "weight"]].to_string())
print()
print("Higher elpd_loo = better predictive accuracy.")
print("The seasonal model should have substantially higher elpd_loo.")

az.plot_compare(comparison, figsize=(8, 3))
plt.title("LOO Comparison: Naive vs Seasonal ITS")
plt.tight_layout()
plt.show()

## Summary

### PPC Diagnostic Workflow

| Check | Tool | Pass Criterion |
|-------|------|---------------|
| Visual PPC | Band overlay | ~94% obs within 94% band |
| Mean | Bayesian p-value | 0.05 < p < 0.95 |
| Variance | Bayesian p-value | 0.05 < p < 0.95 |
| Extremes (min/max) | Bayesian p-value | 0.05 < p < 0.95 |
| Autocorrelation | Bayesian p-value | 0.05 < p < 0.95 |
| Model comparison | LOO-CV | Higher ELPD = preferred model |

### Key Insight

If the observed statistic falls in the extreme tail of the PPC distribution (Bayesian p-value near 0 or 1), the model systematically misses that feature of the data. Common failures:
- **Low lag-1 autocorrelation p-value:** Residual autocorrelation — add AR(1) errors
- **Extreme min/max values:** Heavy tails — consider Student-t likelihood
- **Poor seasonal pattern capture:** Misspecified seasonality — try month dummies vs Fourier

### What's Next
**Module 03:** Synthetic Control Methods — when ITS does not have a valid comparison and you need to construct one from donor units.

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])